# GNN Sinerji Grafı — Explore

Pipeline:
1. `fbref_lineups.run_league_lineups(...)` — maç + lineup verisi
2. `edge_builder.build_edges(season)` — ikili agregatlar
3. `build.build_hetero_graph(season)` — PyG HeteroData
4. `train.train(season)` — model + metrikler
5. `recommend.SynergyEngine` — inference

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from src.collectors import fbref_lineups
from src.graph import edge_builder, build, train, recommend


In [ ]:
# 1. Lineup scrape — önce Süper Lig 2024-25 ile sanity
fbref_lineups.run_league_lineups('TUR-Super Lig', '2425')


In [ ]:
# 2. Edge agregatları
edges = edge_builder.build_edges('2024-2025')
edges['teammate'].nlargest(10, 'minutes_together')


In [ ]:
# 3. PyG graph
data = build.build_hetero_graph('2024-2025')
print(data)


In [ ]:
# 4. Training
train.train('2024-2025', epochs=80)


In [ ]:
# 5. Squad completion sanity — Real Madrid'den Vinicius'u çıkar, sol kanat öner
eng = recommend.SynergyEngine.load('2024-2025')
recs = eng.recommend_squad(
    formation='4-3-3',
    filled={'GK': 'thibaut_courtois', 'LB': 'ferland_mendy', 'CAM': 'jude_bellingham'},
    top_k=5,
)
for pos, lst in recs.items():
    print(f'\n{pos}:')
    for r in lst:
        print(f'  {r.player_name:30s} ({r.team:20s}) score={r.score:.3f}')
